# Semantic Cache — Hands-On

**TechBot scenario:** Users ask TechBot the same questions rephrased slightly. Instead of calling the LLM every time, store responses and return them for semantically similar queries.

**Prerequisites:**
```bash
docker run -d --name redis-stack -p 6379:6379 -p 8001:8001 redis/redis-stack:latest
pip install redisvl sentence-transformers langchain-redis redis
```

## Part 1: Exact Key Cache (SHA256 Hash)

In [ ]:
import redis
import hashlib
import json
import time

r = redis.Redis(host="localhost", port=6379, decode_responses=True)

class ExactCache:
    """Exact-match cache: only byte-identical prompts get a cache hit."""
    
    def __init__(self, redis_client, prefix="llm:exact:", ttl=3600):
        self.r = redis_client
        self.prefix = prefix
        self.ttl = ttl
        self.hits = 0
        self.misses = 0
    
    def _key(self, prompt: str, model: str) -> str:
        content = f"{model}:{prompt}"
        return self.prefix + hashlib.sha256(content.encode()).hexdigest()
    
    def get(self, prompt: str, model: str = "llama3") -> str | None:
        cached = self.r.get(self._key(prompt, model))
        if cached:
            self.hits += 1
            self.r.expire(self._key(prompt, model), self.ttl)  # sliding TTL
            return json.loads(cached)
        self.misses += 1
        return None
    
    def set(self, prompt: str, response: str, model: str = "llama3"):
        self.r.setex(self._key(prompt, model), self.ttl, json.dumps(response))
    
    @property
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0.0


cache = ExactCache(r)

def simulate_llm_call(prompt: str, delay_ms: int = 500) -> str:
    """Fake LLM — sleeps to simulate network + inference latency."""
    time.sleep(delay_ms / 1000)
    return f"[LLM Response to: '{prompt[:40]}...' after {delay_ms}ms]"


def ask(prompt: str, model: str = "llama3") -> str:
    start = time.perf_counter()
    cached = cache.get(prompt, model)
    if cached:
        elapsed = (time.perf_counter() - start) * 1000
        print(f"  CACHE HIT  ({elapsed:.1f}ms): {cached[:60]}")
        return cached
    
    # Cache miss — call the LLM
    response = simulate_llm_call(prompt)
    cache.set(prompt, response, model)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  CACHE MISS ({elapsed:.0f}ms): {response[:60]}")
    return response


print("Exact Cache Demo:")
print("=" * 60)

# First call — cache miss
print("1. First call (cold):")
ask("How do I reset my password?")

# Identical call — cache hit
print("2. Identical call:")
ask("How do I reset my password?")

# Different capitalization — CACHE MISS (exact match only)
print("3. Different capitalization:")
ask("How do I Reset My Password?")

# Rephrased — CACHE MISS
print("4. Rephrased:")
ask("I forgot my password, what should I do?")

print(f"\nHit rate: {cache.hit_rate:.0%} ({cache.hits}/{cache.hits+cache.misses})")
print("→ Exact cache only helps for truly identical prompts (batch jobs, repeated scheduled calls)")

## Part 2: Semantic Cache with redisvl

Now paraphrases of the same question will get cache hits.

In [ ]:
from redisvl.extensions.llmcache import SemanticCache

REDIS_URL = "redis://localhost:6379"

# Create the semantic cache
# Internally: creates a FLAT vector index (brute force — good for < 100K cache entries)
# Distance is COSINE: 0 = identical, 2 = opposite
sem_cache = SemanticCache(
    name="vllm-cache",              # Redis key prefix
    redis_url=REDIS_URL,
    distance_threshold=0.15,        # tune this: lower = stricter, higher = looser
    ttl=3600,                       # 1 hour expiry
)

print("Semantic cache created.")
print(f"Distance threshold: {sem_cache.distance_threshold}")

In [ ]:
def ask_semantic(prompt: str) -> str:
    start = time.perf_counter()
    
    # Check cache first
    results = sem_cache.check(prompt=prompt)
    
    if results:
        elapsed = (time.perf_counter() - start) * 1000
        dist = results[0].get('vector_distance', 'N/A')
        print(f"  CACHE HIT  ({elapsed:.1f}ms, dist={dist:.4f}): {results[0]['response'][:60]}")
        return results[0]['response']
    
    # Cache miss — call LLM
    response = simulate_llm_call(prompt)
    sem_cache.store(prompt=prompt, response=response)
    elapsed = (time.perf_counter() - start) * 1000
    print(f"  CACHE MISS ({elapsed:.0f}ms): {response[:60]}")
    return response


print("Semantic Cache Demo (threshold=0.15):")
print("=" * 60)

# Seed the cache
print("1. First call (cold):")
ask_semantic("How do I reset my password?")

print("2. Different capitalization (exact match → hit):")
ask_semantic("How do I Reset My Password?")

print("3. Rephrased (semantic match → hit):")
ask_semantic("I forgot my password, what should I do?")

print("4. Another rephrase:")
ask_semantic("Can you help me recover access to my account?")

print("5. Completely different topic (should MISS):")
ask_semantic("How much does the Pro plan cost?")

print("6. Install question (should MISS):")
ask_semantic("How do I install the SDK?")

print("7. Install rephrase (should HIT):")
ask_semantic("What command do I use to install the SDK?")

## Part 3: Threshold Tuning — Find the Right Balance

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def cosine_distance(text1: str, text2: str) -> float:
    """Compute cosine distance between two texts (0=identical, 2=opposite)."""
    v1 = embed_model.encode(text1).astype(np.float32)
    v2 = embed_model.encode(text2).astype(np.float32)
    # Cosine similarity → distance
    sim = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    return 1.0 - sim  # distance = 1 - similarity

anchor = "How do I reset my password?"

test_pairs = [
    ("How do I reset my password?",              "IDENTICAL"),
    ("How do I Reset My Password?",              "Same, diff case"),
    ("I forgot my password, what do I do?",      "Paraphrase"),
    ("Can you help me recover my account access?","Loose paraphrase"),
    ("Password recovery instructions",            "Keyword form"),
    ("How do I change my email address?",         "Related but different"),
    ("What is the price of TechBot Pro?",         "Unrelated"),
    ("How do I install the SDK on Windows?",      "Completely different"),
]

print(f"Anchor: '{anchor}'")
print("=" * 70)
print(f"{'Query':<45} {'Distance':>10}  {'Should hit at 0.15?'}")
print("-" * 70)
for query, label in test_pairs:
    dist = cosine_distance(anchor, query)
    would_hit = "YES" if dist <= 0.15 else "no"
    print(f"{query[:44]:<45} {dist:>10.4f}  {would_hit}  ({label})")

In [ ]:
# Visualize the threshold effect
print("\nThreshold sensitivity analysis:")
print("=" * 50)
thresholds = [0.05, 0.10, 0.15, 0.20, 0.25]

for threshold in thresholds:
    hits = sum(1 for q, _ in test_pairs if cosine_distance(anchor, q) <= threshold)
    total = len(test_pairs)
    print(f"  Threshold {threshold:.2f}: {hits}/{total} queries would get cache hits")

print("\nConclusion:")
print("  0.10-0.15: catches obvious paraphrases, low false positive risk → recommended")
print("  0.20-0.25: catches topic-level similarity → risk of wrong cached answers")

## Part 4: LangChain RedisSemanticCache

If you're using LangChain, use `RedisSemanticCache` instead — it hooks directly into the LangChain LLM caching layer.

In [ ]:
# Note: RedisSemanticCache requires an embedding model from LangChain
# We use HuggingFace embeddings (no API key needed)
try:
    from langchain_redis import RedisSemanticCache
    from langchain_community.embeddings import HuggingFaceEmbeddings
    import langchain

    hf_embeddings = HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"}
    )

    semantic_cache = RedisSemanticCache(
        embeddings=hf_embeddings,
        redis_url=REDIS_URL,
        distance_threshold=0.15,
        ttl=3600,
        name="langchain-llmcache"
    )

    # Set as the global LangChain cache
    # After this, any llm.invoke() call is automatically cached
    langchain.globals.set_llm_cache(semantic_cache)

    print("LangChain RedisSemanticCache configured as global cache.")
    print("Now any ChatOpenAI / ChatAnthropic call goes through Redis first.")
    print("")
    print("Example usage (requires API key):")
    print("  from langchain_openai import ChatOpenAI")
    print("  llm = ChatOpenAI(model='gpt-4o-mini')")
    print("  llm.invoke('How do I reset my password?')  # miss → calls OpenAI")
    print("  llm.invoke('I forgot my password')          # hit → from Redis")

except ImportError as e:
    print(f"Install langchain-redis and langchain-community: {e}")

## Part 5: Cache Management

In [ ]:
# Inspect what's in the semantic cache
print("Keys in SemanticCache (vllm-cache):")
r_raw = redis.Redis(host="localhost", port=6379, decode_responses=True)
count = 0
for key in r_raw.scan_iter("vllm-cache*"):
    key_type = r_raw.type(key)
    ttl = r_raw.ttl(key)
    if count < 5:  # show first 5
        print(f"  {key[:60]}  [{key_type}]  TTL={ttl}s")
    count += 1
print(f"  ... ({count} total keys)")

# Adjust threshold dynamically
sem_cache.set_threshold(0.20)
print(f"\nThreshold updated to: {sem_cache.distance_threshold}")

# Clear all cache entries
sem_cache.clear()
print("Cache cleared.")

# Verify
remaining = list(r_raw.scan_iter("vllm-cache*"))
print(f"Keys remaining after clear: {len(remaining)}")

## Part 6: Cost Savings Calculator

In [ ]:
# Model the cost savings from semantic caching

llm_cost_per_call = 0.003        # $0.003 per LLM call (rough average)
llm_latency_ms = 2000            # 2 second typical inference
cache_latency_ms = 3             # 3ms cache lookup
requests_per_hour = 10_000

for hit_rate in [0.30, 0.50, 0.70, 0.85]:
    cache_hits = int(requests_per_hour * hit_rate)
    llm_calls = requests_per_hour - cache_hits
    
    cost_with_cache = llm_calls * llm_cost_per_call
    cost_without_cache = requests_per_hour * llm_cost_per_call
    savings = cost_without_cache - cost_with_cache
    
    avg_latency = (cache_hits * cache_latency_ms + llm_calls * llm_latency_ms) / requests_per_hour
    
    print(f"Hit rate {hit_rate:.0%}:  "
          f"${cost_with_cache:.2f}/hr  "
          f"(saves ${savings:.2f}/hr = ${savings*24*30:.0f}/month)  "
          f"avg latency={avg_latency:.0f}ms")

print(f"\nWithout cache:  ${requests_per_hour * llm_cost_per_call:.2f}/hr  "
      f"avg latency={llm_latency_ms}ms")

## Cleanup

In [ ]:
# Clean up all cache keys
for pattern in ["llm:exact:*", "vllm-cache*", "langchain-llmcache*"]:
    keys = list(r_raw.scan_iter(pattern))
    if keys:
        r_raw.delete(*keys)
        print(f"Deleted {len(keys)} keys matching '{pattern}'")

## Summary

| Cache Type | Hit Condition | False Positive Risk | Use Case |
|------------|--------------|--------------------|---------|
| Exact (SHA256) | Byte-identical prompt | Zero | Batch jobs, deterministic pipelines |
| Semantic (redisvl) | Cosine distance ≤ threshold | Low (with threshold 0.15) | Production chatbots |
| LangChain global | Same as semantic | Same | LangChain-based apps |

**Threshold rule of thumb:**
- Start at 0.15, measure actual hit rate after 24h
- If users report wrong answers → lower to 0.10
- If hit rate is < 20% → raise to 0.20 carefully

Next: **04_celery_flower/02_hands_on.ipynb** — Handle concurrent users without blocking.